# Analizador cuantitativo Crash/Boom (solo lectura)

Este cuaderno usa el módulo `boom_crash_quant_analyzer.py` para recopilar datos públicos de Deriv, estudiar eventos extremos candidatos y analizar zonas históricas de reacción. No ejecuta operaciones ni garantiza que un evento estadístico sea un spike oficial.

**Endpoint público configurado:** `wss://api.derivws.com/trading/v1/options/ws/public`, recomendado por la guía principal actual de Deriv (`llms.txt`). No requiere App ID, OTP ni token para datos públicos. El cliente intenta el endpoint público actual y tiene un fallback de compatibilidad con el endpoint de market-data de la guía oficial. También suprime el Origin HTTP artificial del cliente de Python. El cuaderno reintenta fallos de transporte temporales, pero ninguna conexión puede garantizarse frente a caídas externas persistentes.

Documentación oficial: [llms.txt](https://developers.deriv.com/llms.txt) · [API overview](https://developers.deriv.com/docs/intro/api-overview/) · [datos de mercado](https://developers.deriv.com/docs/workflows/)

**Importante:** las tasas calculadas son exploratorias hasta que se validen temporalmente fuera de muestra.

In [ ]:
%pip install -q websocket-client pandas numpy plotly scipy statsmodels arch
import logging, os, sys
os.environ["DERIV_MARKET_WS_URL"] = "wss://api.derivws.com/trading/v1/options/ws/public"
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
print('Dependencias instaladas. Python:', sys.version.split()[0])
print('Endpoint público configurado:', os.environ["DERIV_MARKET_WS_URL"])

In [ ]:
# Sube el módulo .py correcto; comprobar el nombre del módulo, NO el del cuaderno.
from pathlib import Path
import importlib, sys

MODULE_FILE = Path('/content/boom_crash_quant_analyzer.py')
CURRENT_MARKER = 'MODULE_VERSION = "2.2.0"'

if not MODULE_FILE.exists() or CURRENT_MARKER not in MODULE_FILE.read_text(encoding='utf-8'):
    print('Sube el archivo corregido boom_crash_quant_analyzer.py (versión 2.2.0).')
    from google.colab import files
    uploaded = files.upload()
    if not MODULE_FILE.exists() or CURRENT_MARKER not in MODULE_FILE.read_text(encoding='utf-8'):
        raise FileNotFoundError(
            'No se encontró el módulo boom_crash_quant_analyzer.py versión 2.2.0. '
            'Comprueba que subiste el archivo con ese nombre exacto.'
        )

if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import boom_crash_quant_analyzer as analyzer_module
analyzer_module = importlib.reload(analyzer_module)
DerivMarketDataClient = analyzer_module.DerivMarketDataClient
SyntheticIndexAnalyzer = analyzer_module.SyntheticIndexAnalyzer
AnalyzerConfig = analyzer_module.AnalyzerConfig
print('Módulo importado. Versión:', analyzer_module.MODULE_VERSION)

In [ ]:
# Configuración de reproducibilidad
import numpy as np
import random
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print(f'Seeds fijados: numpy={SEED}, random={SEED}')

## 1. Instanciar cliente y analizar Boom 1000
Crea el cliente de mercado (una sola vez) y pásalo a cada analizador para reutilizar la conexión.

In [ ]:
market_client = DerivMarketDataClient()
boom = SyntheticIndexAnalyzer('Boom 1000', direction='up', days=20, client=market_client)
boom_candles = boom.fetch_historical_data()
boom_ticks = boom.fetch_tick_history(count_limit=5000)
boom_tfs = [tf for tf in ['1min', '5min', '15min', '1h', '4h', '1d'] if tf in boom.data]
boom_results = boom.full_analysis(boom_tfs)
if len(boom_ticks) >= 500: boom.detect_tick_extremes()
print('Símbolo confirmado:', boom.symbol, '| nombre:', boom.resolved_name)
for tf, result in boom_results.items():
    print(f'{tf}: eventos={len(result["events"])}, zonas={len(result["zones"])}')

## 2. Visualizaciones completas — Boom 1000
Todos los gráficos solicitados con las nuevas mejoras estadísticas.

In [ ]:
# Gráfico de ticks con detección de spikes
try:
    boom.plot_tick_spikes(threshold_quantile=0.999, window_ticks=2000).show()
except Exception as exc:
    print('Gráfico de spikes no disponible:', exc)

# Velas OHLC con zonas técnicas y eventos
if '5min' in boom.data:
    boom.plot_candles_with_zones('5min').show()

In [ ]:
# Distribución de retornos y colas extremas (con EVT/GPD)
try:
    boom.plot_return_distribution(timeframe='ticks', fit_evt=True, threshold_quantile=0.99).show()
except Exception as exc:
    print('Distribución EVT no disponible:', exc)

In [ ]:
# Volatilidad móvil y regímenes de volatilidad (Markov-Switching / CUSUM)
if '5min' in boom.data:
    try:
        boom.plot_volatility('5min', rolling_window=20, show_regimes=True).show()
    except Exception as exc:
        print('Regímenes de volatilidad no disponibles:', exc)

In [ ]:
# Distribución del intervalo entre spikes y curva de supervivencia (Kaplan-Meier)
try:
    boom.plot_interevent_survival(timeframe='ticks', min_events=10).show()
except Exception as exc:
    print('Curva de supervivencia no disponible:', exc)

In [ ]:
# Autocorrelación de retornos y de retornos al cuadrado (volatility clustering)
try:
    boom.plot_autocorrelation(timeframe='ticks', max_lag=50).show()
except Exception as exc:
    print('Autocorrelación no disponible:', exc)

In [ ]:
# Mapa de calor de frecuencia y zonas de precio
if '5min' in boom.data:
    try:
        boom.plot_price_heatmap('5min').show()
    except Exception as exc:
        print('Heatmap precio-hora no disponible:', exc)

In [ ]:
# Gráfico de probabilidades condicionadas
try:
    boom.plot_conditional_event_probability(horizon_ticks=100, bin_size=100).show()
except Exception as exc:
    print('Probabilidad condicionada no disponible:', exc)

In [ ]:
# Diagnósticos estadísticos (Ljung-Box, Jarque-Bera, ARCH-LM, ADF)
try:
    diag = boom.residual_diagnostics(timeframe='ticks', max_lag=20)
    print('Diagnósticos residuales (ticks):')
    for k, v in diag.items():
        print(f'  {k}: {v}')
except Exception as exc:
    print('Diagnósticos no disponibles:', exc)

In [ ]:
# Ajuste EVT/GPD explícito
try:
    evt = boom.fit_evt_gpd(timeframe='ticks', threshold_quantile=0.99, direction='directional')
    print('EVT/GPD Fit (ticks, directional):')
    for k, v in evt.items():
        if k != 'mean_excess_plot':
            print(f'  {k}: {v}')
except Exception as exc:
    print('EVT no disponible:', exc)

In [ ]:
# Detección de regímenes de volatilidad
if '5min' in boom.data:
    try:
        regimes = boom.detect_volatility_regimes('5min', n_regimes=2)
        print('Regímenes de volatilidad (5min):')
        for k, v in regimes.items():
            print(f'  {k}: {v}')
    except Exception as exc:
        print('Regímenes no disponibles:', exc)

In [ ]:
# GARCH(1,1) forecasting
try:
    garch = boom.fit_garch(timeframe='ticks', p=1, q=1)
    print('GARCH(1,1) Fit (ticks):')
    for k, v in garch.items():
        if k not in ['conditional_volatility', 'forecast_horizon_10']:
            print(f'  {k}: {v}')
    if 'forecast_horizon_10' in garch:
        print(f'  Forecast 10-step: {garch["forecast_horizon_10"]}')
except Exception as exc:
    print('GARCH no disponible:', exc)

## 3. Validación Walk-Forward (Out-of-Sample)
Framework de validación temporal para evitar sesgo retrospectivo.

In [ ]:
# Walk-forward validation en 5min
if '5min' in boom.data:
    try:
        wf_results = boom.walk_forward_validation(
            timeframe='5min',
            train_window_days=60,
            test_window_days=14,
            step_days=7,
            min_train_bars=200
        )
        print(f'Walk-forward folds: {len(wf_results)}')
        if not wf_results.empty:
            print(wf_results[['fold', 'zone_type', 'zone_price', 'oos_evaluated',
                                'oos_reaction_rate_pct', 'oos_p_value_vs_random',
                                'oos_reaction_ci_lower_pct', 'oos_reaction_ci_upper_pct']].to_string(index=False))
            agg = boom.aggregate_walk_forward_results(wf_results)
            print('\nAgregación Walk-Forward:')
            for k, v in agg.items():
                if k != 'folds_summary':
                    print(f'  {k}: {v}')
    except Exception as exc:
        print('Walk-forward no disponible:', exc)

## 4. Ejecutar el análisis independiente de Crash 500
La instancia Crash usa dirección bajista y mantiene completamente separado el estado de Boom. Ejecuta esta celda solo cuando hayas revisado que la primera instancia funciona.

In [ ]:
crash = SyntheticIndexAnalyzer('Crash 500', direction='down', days=20, client=market_client)
crash_candles = crash.fetch_historical_data()
crash_ticks = crash.fetch_tick_history(count_limit=5000)
crash_tfs = [tf for tf in ['1min', '5min', '15min', '1h', '4h', '1d'] if tf in crash.data]
crash_results = crash.full_analysis(crash_tfs)
if len(crash_ticks) >= 500: crash.detect_tick_extremes()
print('Símbolo confirmado:', crash.symbol, '| nombre:', crash.resolved_name)
for tf, result in crash_results.items():
    print(f'{tf}: eventos={len(result["events"])}, zonas={len(result["zones"])}')
if '5min' in crash.data:
    crash.plot_candles_with_zones('5min').show()
    crash.plot_event_heatmap('5min', metric='count').show()

## 5. Exportar tablas CSV
Los archivos quedan en el entorno temporal de Colab. Descárgalos antes de terminar la sesión para no perderlos.

In [ ]:
import zipfile
from pathlib import Path
boom_files = boom.export_csv('analysis_output/boom1000')
crash_files = crash.export_csv('analysis_output/crash500')
all_files = boom_files + crash_files
archive = Path('crash_boom_analysis_csv.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for filename in all_files: zf.write(filename, arcname=filename)
print('CSV exportados:', len(all_files), '| ZIP:', archive)
from google.colab import files
files.download(str(archive))